### PyTorch AlexNet Exercises

Welcome to the PyTorch AlexNet exercise template notebook.

There are several questions in this notebook and it's your goal to answer them by writing Python and PyTorch code.






In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np

# Residual Block (BasicBlock for ResNet-18/34)
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super(BasicBlock, self).__init__()
        # Main layers definition (F(x))
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        # Shortcut definition (Identity x)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion * planes:
            # If dimensions change, use 1x1 Conv to match
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion * planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion * planes)
            )

    def forward(self, x):
        # H(x) = F(x) + x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        
        out += self.shortcut(x) # Add the shortcut (Skip Connection)
        out = F.relu(out)
        return out

# ResNet-18 Architecture
class ResNet(nn.Module):
    def __init__(self, block, num_blocks, num_classes=200):
        super(ResNet, self).__init__()
        self.in_planes = 64
        
        # Initial Convolutional Layer
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        
        # Layer Stages (Stage 1-4)
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)
        
        # Final fully connected layer
        self.linear = nn.Linear(512 * block.expansion, num_classes)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks - 1)
        layers = []
        for current_stride in strides:
            layers.append(block(self.in_planes, planes, current_stride))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        
        # Global Average Pooling 
        out = F.adaptive_avg_pool2d(out, (1, 1))
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return out

def ResNet18():
    # ResNet-18 structure: [2, 2, 2, 2] blocks per stage
    return ResNet(BasicBlock, [2, 2, 2, 2])

# Hyperparameters
learning_rates = [0.1, 0.001, 0.0001]
batch_sizes = [16, 32, 64]
num_epochs = 10
results = {}

# Define transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Load the Tiny ImageNet dataset (Example path)
train_dataset = datasets.ImageFolder('/path/to/tiny-imagenet/train', transform=transform)
val_dataset = datasets.ImageFolder('/path/to/tiny-imagenet/val', transform=transform)


for lr in learning_rates:
    for batch_size in batch_sizes:
        print(f'\n--- Configuration: LR={lr}, Batch Size={batch_size} ---')
        
        # Data loaders
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

        # Initialize ResNet-18 Network
        net = ResNet18().cuda() 

        # Loss and optimizer
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.SGD(net.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)

        for epoch in range(1, num_epochs + 1):
            
            # --- TRAINING LOOP ---
            train_loss = 0.0
            net.train()
            with tqdm(train_loader, unit="batch", leave=False) as tepoch:
                tepoch.set_description(f"Epoch {epoch}/{num_epochs} [Train]")
                for inputs, labels in tepoch:
                    inputs, labels = inputs.cuda(), labels.cuda()
                    optimizer.zero_grad()
                    outputs = net(inputs)
                    loss = criterion(outputs, labels)
                    loss.backward()
                    optimizer.step()
                    train_loss += loss.item() * inputs.size(0)
                    tepoch.set_postfix(loss=loss.item())

            # --- VALIDATION LOOP ---
            val_loss = 0.0
            correct = 0
            total = 0
            net.eval()
            with torch.no_grad(), tqdm(val_loader, unit="batch", leave=False) as tepoch:
                tepoch.set_description(f"Epoch {epoch}/{num_epochs} [Val]")
                
                for inputs, labels in tepoch:
                    inputs, labels = inputs.cuda(), labels.cuda()
                    outputs = net(inputs)
                    loss = criterion(outputs, labels)
                    val_loss += loss.item() * inputs.size(0)
                    _, predicted = torch.max(outputs.data, 1)
                    total += labels.size(0)
                    correct += (predicted == labels).sum().item()
            
            # Final calculations for the epoch
            train_loss /= len(train_loader.dataset)
            val_loss /= len(val_loader.dataset)
            val_accuracy = 100 * correct / total

            print(f'Epoch [{epoch}/{num_epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.2f}%')

        # Saving final accuracy for comparison
        results[(lr, batch_size)] = val_accuracy


# Final Report
print("\n\n" + "="*50)
print("FINAL RESNET-18 HYPERPARAMETER SEARCH SUMMARY")
print("="*50)

for params, accuracy in results.items():
    lr, batch_size = params
    print(f'LR: {lr:<6}, Batch Size: {batch_size:<3}, Final Val. Accuracy: {accuracy:.2f}%')

print("="*50)